In [1]:
# Copyright (c) Microsoft. All rights reserved.

import asyncio

from azure.identity.aio import DefaultAzureCredential

from semantic_kernel.agents import AgentGroupChat, AzureAIAgent, AzureAIAgentSettings, ChatCompletionAgent
from semantic_kernel.agents.strategies import TerminationStrategy
from semantic_kernel.connectors.ai.open_ai import AzureChatCompletion
from semantic_kernel.contents import AuthorRole
from dotenv import load_dotenv
import os
from azure.ai.projects.aio import AIProjectClient

In [12]:
project_client = AzureAIAgent.create_client(credential=DefaultAzureCredential(), conn_str=os.getenv("AZURE_AI_CONNECTION_STRING"))
agent_id = "asst_BFI3WFBIbnNOtSiuz0zZm5h5"

In [ ]:
product_information_agent = await project_client.agents.get_agent(agent_id)
prod_sk_agent = AzureAIAgent(client=project_client, definition=product_information_agent)

In [6]:
import asyncio
import os
import sys
from datetime import datetime

from semantic_kernel.agents import ChatCompletionAgent, ChatHistoryAgentThread
from semantic_kernel.filters import FunctionInvocationContext
from semantic_kernel.connectors.ai import FunctionChoiceBehavior
from semantic_kernel.connectors.ai.open_ai import AzureChatCompletion
from semantic_kernel.functions import KernelArguments
from semantic_kernel.kernel import Kernel
from azure.identity import DefaultAzureCredential, get_bearer_token_provider
from dotenv import load_dotenv
import pyodbc
import aioodbc
from semantic_kernel.functions import kernel_function
from typing import Annotated, Optional
import pandas as pd
import json

driver = os.environ["SQL_DRIVER"]
server = os.environ["SQL_SERVER"]
database = os.environ["SQL_DATABASE"]
user = os.environ["SQL_USERNAME"]
password = os.environ["SQL_PASSWORD"]
conn_string = f"Driver={driver};Server={server},1433;Database={database};Uid={user};Pwd={password};Encrypt=yes;TrustServerCertificate=no;Connection Timeout=60;"
table_name = "product_pricing"
conn = pyodbc.connect(conn_string)

column_info = []
cur = conn.cursor()
cur.execute(f"SELECT COLUMN_NAME, DATA_TYPE FROM INFORMATION_SCHEMA.COLUMNS WHERE TABLE_NAME = '{table_name}'")
columns = cur.fetchall()
    # col[1] is the column name, col[2] is the column type
column_info = [f"{col[0]}: {col[1]}" for col in columns]

cur = conn.cursor()
cur.execute(f"SELECT DISTINCT category, product_name from {table_name}")
columns = cur.fetchall()
    # col[1] is the column name, col[2] is the column type
unique_product_category = [f"{col[0]}: {col[1]}" for col in columns]

table_dicts = []
table_dicts.append({"table_name": table_name, "column_names": column_info})

database_info = "\n".join(
    [
        f"Table {table['table_name']} Schema: Columns: {', '.join(table['column_names'])}"
        for table in table_dicts
    ]
)
products = unique_product_category

database_info += f"\n Unique category: products combination - {', '.join(unique_product_category)}"
database_info += "\n\n"

class PricingPlugin:
   
    """A sample Menu Plugin used for the concept sample."""

    @kernel_function(description="This function is used to answer user questions about product pricing and inventory by executing SQL queries against the database.")
    async def execute_sql(self, sql_query: Annotated[str, "The input should be a well-formed SQL query to extract information based on the user's question. The query result will be returned as a JSON object."]) -> Annotated[str, "Return data in JSON serializable format"]:
        try:
            conn = await aioodbc.connect(dsn=conn_string)
            # Perform the query asynchronously
            cur = await conn.cursor()
            await cur.execute(sql_query)
            rows = await cur.fetchall()
            columns = [description[0] for description in cur.description]

            if not rows:  # No need to create DataFrame if there are no rows
                return json.dumps("The query returned no results. Try a different question.")
            results = [tuple(row) for row in rows]
            await conn.close()
            data = pd.DataFrame(results, columns=columns)
            # return data.to_dict(orient="records")
            return data.to_json(index=False, orient="split")

        except Exception as e:
            return json.dumps({"SQL query failed with error": str(e), "query": sql_query})



kernel = Kernel()
load_dotenv()
# Add the AzureChatCompletion AI Service to the Kernel
service_id = "openai"
credential = DefaultAzureCredential()
token_provider = get_bearer_token_provider(
    credential, "https://cognitiveservices.azure.com/.default"
)
chat_completion = AzureChatCompletion(ad_token_provider=token_provider, deployment_name=os.environ["AOAI_DEPLOYMENT_NAME"], endpoint=os.environ["AOAI_ENDPOINT"], api_version=os.environ["AOAI_API_VERSION"], service_id=service_id)
kernel.add_service(chat_completion)

settings = kernel.get_prompt_execution_settings_from_service_id(service_id=service_id)
# Configure the function choice behavior to auto invoke kernel functions
settings.function_choice_behavior = FunctionChoiceBehavior.Auto()

# Create the agent
pricing_agent = ChatCompletionAgent(
    kernel=kernel,
    name="PRICING_AGENT",
    instructions=f"""
        You are a pricing agent. Your job is to provide pricing information for a given product. 
        Use the pricing database as defined by the schema: {database_info}
        All product names are lowercase with spaces replaced by underscores. 
        """,
    arguments=KernelArguments(settings=settings),
    plugins=[PricingPlugin()],
)

Unclosed client session
client_session: <aiohttp.client.ClientSession object at 0x000001AB3E5AC700>
Unclosed client session
client_session: <aiohttp.client.ClientSession object at 0x000001AB3E5AEA70>


In [26]:
chat = AgentGroupChat(agents=[prod_sk_agent, pricing_agent])

In [27]:
input = "What tent would you recommend for a rainy day and how much would it cost?"
await chat.add_chat_message(input)
print(f"# {AuthorRole.USER}: '{input}'")
async for content in chat.invoke():
    print(f"# {content.role} - {content.name or '*'}: '{content.content}'")

# 5. Done and remove the Auzre AI Foundry Agent.
print(f"# IS COMPLETE: {chat.is_complete}")

# user: 'What tent would you recommend for a rainy day and how much would it cost?'
# assistant - RESEARCHER: 'I recommend considering the "All-Weather Tent" for a rainy day. It features waterproof fabric to withstand heavy rain, reinforced aluminium poles for wind resistance, mesh ventilation for airflow, and UV-resistant material. This tent is available in sizes for 2-person, 4-person, and 6-person accommodations, making it ideal for couples, families, or small groups【3:0†source】.

As for the cost, you would need to look at the specific retailers or online marketplaces to find the exact pricing for each size, as costs can vary based on location and availability.'
# assistant - PRICING_AGENT: 'The prices for the "All-Weather Tent" are as follows:

- **2-person tent:** $150 USD
- **4-person tent:** $220 USD
- **6-person tent:** $280 USD

You can choose the size that best fits your needs and budget.'
# assistant - RESEARCHER: 'I recommend considering the "All-Weather Tent" for a rainy d